# Python Concurrency — Expert Interview Guide

Covers: GIL, threading, multiprocessing, asyncio, executors.

| | CPU-bound | I/O-bound |

|---|---|---|

| One thread | Normal code | Normal code |

| Multi | multiprocessing | threading / asyncio |

## 1. The GIL (Global Interpreter Lock)

The GIL is a mutex in CPython that allows **only one thread to execute Python bytecode at a time**.

- **I/O-bound**: GIL released during waits -> threading helps

- **CPU-bound**: GIL prevents true parallelism -> use multiprocessing

- PyPy, Jython, GraalPy do not have a GIL

In [ ]:
import threading, time

def count_up(n):
    x = 0
    for _ in range(n): x += 1
    return x

N = 5_000_000

start = time.perf_counter()
count_up(N); count_up(N)
seq_time = time.perf_counter() - start

start = time.perf_counter()
t1 = threading.Thread(target=count_up, args=(N,))
t2 = threading.Thread(target=count_up, args=(N,))
t1.start(); t2.start()
t1.join(); t2.join()
thread_time = time.perf_counter() - start

print(f"Sequential: {seq_time:.3f}s")
print(f"Threaded:   {thread_time:.3f}s")
print(f"Threading is NOT faster for CPU-bound (GIL!)")

# I/O-bound: threading DOES help
def io_task(): time.sleep(0.1)

start = time.perf_counter()
for _ in range(4): io_task()
seq_io = time.perf_counter() - start

start = time.perf_counter()
threads = [threading.Thread(target=io_task) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
thread_io = time.perf_counter() - start

print(f"I/O Sequential: {seq_io:.3f}s")
print(f"I/O Threaded:   {thread_io:.3f}s  (~4x faster!)")

> **Interview Insight:** Python 3.13 introduced experimental no-GIL mode (`--disable-gil`). The GIL is a CPython implementation detail, not a Python language requirement.

## 2. Threading — Synchronization Primitives

### `threading.Lock()`

A Lock ensures **only one thread accesses a shared resource at a time**, preventing race conditions.

`counter += 1` looks like one step but is **three CPU instructions** (READ → ADD → WRITE). Without a lock, two threads can interleave these steps and lose increments.

In [ ]:
Thread 1: READ (counter=5)
Thread 2: READ (counter=5)   ← reads before T1 writes back
Thread 1: ADD → 6, WRITE 6
Thread 2: ADD → 6, WRITE 6  ← overwrites T1, one increment lost → race condition

`with lock:` is shorthand for `lock.acquire()` + `lock.release()`. Always use `with` — an exception between acquire/release causes a **deadlock** without it.

---

### `threading.Semaphore(N)`

A Semaphore limits how many threads can enter a block **simultaneously**. Think of it as a **bouncer at a door** — only N threads allowed inside, others wait outside.

`with sem:` is shorthand for `sem.acquire()` + `sem.release()`. It does **not** create threads — it only controls how many can enter at the same time.

In [ ]:
10 threads created → all reach 'with sem:'
Semaphore(3) → only 3 enter → 7 wait outside
One finishes → next one enters → always max 3 inside

> **`t.start()`** = launches the thread — function runs in background, main program continues immediately  

> **`t.join()`** = main program stops and waits until that thread finishes  

> **Main program** = the default thread Python starts with — runs your code top to bottom

In [ ]:
import threading, time

counter = 0
lock = threading.Lock()

def increment_safe(n):
    global counter
    for _ in range(n):
        with lock:
            counter += 1

threads = [threading.Thread(target=increment_safe, args=(1000,)) for _ in range(5)]
for t in threads: t.start()
for t in threads: t.join()
print(f"Safe counter: {counter}")  # always 5000

# Semaphore -- limit concurrent access
sem = threading.Semaphore(3)  # max 3 concurrent

def use_resource(name):
    with sem:
        print(f"  [{name}] using resource")
        time.sleep(0.02)

threads = [threading.Thread(target=use_resource, args=(i,)) for i in range(6)]
for t in threads: t.start()
for t in threads: t.join()

# Event -- signal between threads
ready = threading.Event()

def producer():
    time.sleep(0.05)
    print("[producer] ready")
    ready.set()

def consumer():
    print("[consumer] waiting...")
    ready.wait()
    print("[consumer] proceeding")

p = threading.Thread(target=producer)
c = threading.Thread(target=consumer)
c.start(); p.start()
p.join(); c.join()

# threading.local -- per-thread storage
local = threading.local()

def set_name(name):
    local.name = name
    time.sleep(0.01)
    print(f"  {name} sees: {local.name}")

threads = [threading.Thread(target=set_name, args=(f"W{i}",)) for i in range(3)]
for t in threads: t.start()
for t in threads: t.join()

> **Interview Insight:** Always use `with lock:` over `.acquire()/.release()` — the context manager guarantees release even on exceptions, preventing deadlocks.

## 3. `concurrent.futures` — High-Level API

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def fetch(url_num):
    time.sleep(0.1)
    return f"data-{url_num}"

# map: submit all, get results in order
with ThreadPoolExecutor(max_workers=4) as ex:
    results = list(ex.map(fetch, range(8)))
    print("map:", results[:4])

# submit: get results as completed
with ThreadPoolExecutor(max_workers=4) as ex:
    futures = {ex.submit(fetch, i): i for i in range(4)}
    for future in as_completed(futures):
        url_num = futures[future]
        print(f"  url-{url_num}: {future.result()}")

# Future API
with ThreadPoolExecutor() as ex:
    f = ex.submit(fetch, 99)
    print(f"Done before result: {f.done()}")
    print(f"Result: {f.result(timeout=5)}")
    print(f"Done after result: {f.done()}")

> **Interview Insight:** `executor.map()` returns results in submission order and re-raises exceptions during iteration. `executor.submit()` + `as_completed()` returns results as they finish — better for heterogeneous task durations.

## 4. asyncio — Event Loop & Coroutines

`asyncio` is cooperative multitasking on **one thread**:

- Coroutines (`async def`) yield control with `await`

- Event loop switches between coroutines at `await` points

- Best for I/O-bound concurrent tasks with many connections

In [ ]:
import asyncio, time

async def task(name, delay):
    print(f"  [{name}] start")
    await asyncio.sleep(delay)  # non-blocking
    print(f"  [{name}] done")
    return f"result-{name}"

# asyncio.gather -- run concurrently
async def main_gather():
    start = time.perf_counter()
    results = await asyncio.gather(
        task("A", 0.1),
        task("B", 0.2),
        task("C", 0.1),
    )
    elapsed = time.perf_counter() - start
    print(f"gather: {elapsed:.2f}s (not 0.4s!)")
    print(results)

asyncio.run(main_gather())

# create_task -- fire and forget
async def main_tasks():
    t1 = asyncio.create_task(task("t1", 0.1))
    t2 = asyncio.create_task(task("t2", 0.2))
    print("main: doing other work")
    await asyncio.sleep(0.05)
    r1 = await t1
    r2 = await t2
    print(r1, r2)

asyncio.run(main_tasks())

> **Interview Insight:** If any coroutine calls `time.sleep()` (blocking) instead of `await asyncio.sleep()`, it blocks the **entire event loop** — all other coroutines freeze. Always use async versions of I/O operations.

## 5. asyncio Queue — Producer-Consumer

In [ ]:
import asyncio, random

async def producer(queue, n):
    for i in range(n):
        item = random.randint(1, 50)
        await queue.put(item)
        print(f"  [P] put {item}")
        await asyncio.sleep(0.01)
    await queue.put(None)  # sentinel

async def consumer(queue, name):
    while True:
        item = await queue.get()
        if item is None:
            queue.task_done()
            break
        print(f"  [{name}] got {item}")
        await asyncio.sleep(0.02)
        queue.task_done()

async def main():
    q = asyncio.Queue(maxsize=5)
    await asyncio.gather(
        producer(q, 5),
        consumer(q, "C1"),
    )

asyncio.run(main())

# asyncio.Semaphore -- limit concurrent connections
async def fetch_limited(sem, n):
    async with sem:
        await asyncio.sleep(0.1)
        return f"data-{n}"

async def main_sem():
    sem = asyncio.Semaphore(3)  # max 3 concurrent
    results = await asyncio.gather(*[fetch_limited(sem, i) for i in range(8)])
    print(f"Got {len(results)} results")

asyncio.run(main_sem())

> **Interview Insight:** Use `asyncio.Semaphore` to rate-limit concurrent API calls (e.g., max 10 concurrent HTTP requests). Without it, `asyncio.gather` fires all requests simultaneously, potentially hitting rate limits.

## 6. `run_in_executor` — Mix Sync and Async

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor

def blocking_io(n):
    import time; time.sleep(0.05)
    return f"result-{n}"

async def main():
    loop = asyncio.get_event_loop()

    # Run blocking function without blocking event loop
    with ThreadPoolExecutor(max_workers=4) as pool:
        tasks = [loop.run_in_executor(pool, blocking_io, i) for i in range(4)]
        results = await asyncio.gather(*tasks)
        print("Threaded results:", results)

    # asyncio.to_thread (Python 3.9+) -- simpler
    def sync_fn(): return "sync done"
    result = await asyncio.to_thread(sync_fn)
    print(result)

asyncio.run(main())

# Decision guide
print("Decision guide:")
print("  I/O-bound, many connections -> asyncio")
print("  I/O-bound, simple           -> threading")
print("  CPU-bound                   -> multiprocessing")
print("  Blocking lib in async code  -> run_in_executor")

> **Interview Insight:** `asyncio.to_thread()` (3.9+) is just `loop.run_in_executor(None, fn)` with a nicer API. The default executor is a `ThreadPoolExecutor`. For CPU-bound work in async, use `ProcessPoolExecutor` explicitly.